# CROSS step 3 on Kaggle: resnet50

`erm` and `cross_p_r25` with resnet50 (lr 3e-3), held-out-source protocol, seeds 42-44, 5 folds (30 runs). `code/cross_pilot.py`
also scores the real and edited validation folds (`preds_val/`) for the validation threshold.
**Inputs:** `masoudnickparvar/brain-tumor-mri-dataset` and the private bundle dataset (version 4). **Settings:** GPU T4, Internet on.


In [ ]:
import glob, os, shutil, subprocess
def find(rel):
    # fixed-depth search (a recursive glob over the mounted inputs takes minutes)
    for depth in range(0, 5):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * depth + rel))
        if hits:
            return hits[0]
    raise FileNotFoundError(rel)
bundle = find("results/audit/grouped_folds.csv")[: -len("results/audit/grouped_folds.csv")].rstrip("/")
data = find("Training/glioma")[: -len("Training/glioma")].rstrip("/")   # the folder that holds Training/ and Testing/
assert os.path.isdir(f"{data}/Testing"), data
P = "/kaggle/working/proj"
if not os.path.exists(P):
    shutil.copytree(bundle, P)
os.makedirs(f"{P}/data", exist_ok=True)
if not os.path.exists(f"{P}/data/brain-tumor-mri-dataset"):
    os.symlink(data, f"{P}/data/brain-tumor-mri-dataset")
print("bundle:", bundle, "| data:", data)
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import sys
env = dict(os.environ, CROSS_OUT="/kaggle/working/cross_resnet50", PYTHONUNBUFFERED="1")
MODEL = ["--model", "resnet50", "--lr", "3e-3"]
def go(arms, seeds):
    cmd = [sys.executable, "code/cross_pilot.py", "--arms", arms, "--folds", "0,1,2,3,4", "--seeds", seeds] + MODEL
    p = subprocess.Popen(cmd, cwd=P, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0
# QUICK_TEST: one 1-epoch run to check the setup (writes to a separate folder)
QUICK_TEST = False
if QUICK_TEST:
    env["CROSS_OUT"] = "/kaggle/working/cross_quick"
    subprocess.run([sys.executable, "code/cross_pilot.py", "--arms", "cross_p", "--folds", "0", "--seeds", "1", "--epochs", "1"] + MODEL, cwd=P, env=env, check=True)
else:
    # 30 runs in one process, so each fold's edited validation set is built once
    go("erm,cross_p_r25", "42,43,44")


In [ ]:
shutil.make_archive("/kaggle/working/cross_resnet50_results", "zip", "/kaggle/working/cross_resnet50")
print(open("/kaggle/working/cross_resnet50/results.csv").read())
